# CS2309 — Precision eval (chọn version)

**Không bắt buộc chạy hết list.** Mỗi session chọn 1–vài config → export `bundle.zip` → so sánh local sau.

Hệ quy chiếu jobs: `jobs_june17.json`. Baseline chất lượng khi merge: `baseline_fp32` (chạy riêng cũng được).

| Alias | Config chuẩn |
|---|---|
| `fp32` / `32` | `baseline_fp32` |
| `fp16` / `16` | `improved_fp16_cache` (EditCache) |
| `fp8` / `8` | `improved_fp8_cache` |
| `fp4` / `4` | `improved_fp4_cache` |
| `fp16_weight` / `16_weight` | `fp16_disk` |
| `fp4_weight` / `4_weight` | `fp4_from_fp16` |

### ⓪ Chọn config (chỉ những cái bạn muốn chạy lần này)

In [1]:
# Bật True cho config muốn chạy. Có thể chỉ chọn 1.
SELECT = {
    "fp32": True,          # baseline — nên có 1 lần để PSNR vs FP32 khi merge
    "fp16": False,          # improved_fp16_cache
    "fp8": False,           # improved_fp8_cache (T4 hay fail chất lượng)
    "fp4": False,           # improved_fp4_cache
    "fp16_weight": False,    # disk fp16
    "fp4_weight": False,     # disk fp16 → quant fp4
}
EVAL_CONFIGS = ",".join(k for k, on in SELECT.items() if on)
assert EVAL_CONFIGS, "Bật ít nhất 1 config trong SELECT"

USE_DRIVE = False
USE_PRIVATE_REPO = True
REPO_SLUG = "NguyenKz/CS2309.CH201"
COLAB_REPO_DIR = "/content/CS2309.CH201"
# Prefer Drive (~5GB fp16) — KHÔNG kéo Qualcomm 10GB+ rồi xóa archive
DRIVE_FP16 = "/content/drive/MyDrive/CS2309/swiftedit_weights_fp16"
DRIVE_FP32 = "/content/drive/MyDrive/CS2309/swiftedit_weights"
# True: setup_colab bỏ tải Qualcomm; weights do prepare_colab_weights (Drive trước)
SKIP_QUALCOMM_IN_SETUP = True
# True: thiếu Drive fp16 → không convert trên Colab (fail sớm, khuyên upload từ Mac)
NO_CONVERT_ON_COLAB = True

# Smoke: MAX_JOBS=6. Full cùng June17: MAX_JOBS=600
N_IMAGES = 2
EDITS_PER_IMAGE = 3
MAX_JOBS = 6
print("EVAL_CONFIGS =", EVAL_CONFIGS)
print("MAX_JOBS =", MAX_JOBS)
print("SKIP_QUALCOMM_IN_SETUP =", SKIP_QUALCOMM_IN_SETUP)

EVAL_CONFIGS = fp32
MAX_JOBS = 6
SKIP_QUALCOMM_IN_SETUP = True


### ① Clone + GPU + token

In [2]:
import getpass, os, subprocess, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

COLAB_REPO_DIR = Path(COLAB_REPO_DIR)
DRIVE_FP16 = Path(DRIVE_FP16)

def _colab_repo_url():
    if not USE_PRIVATE_REPO:
        return f"https://github.com/{REPO_SLUG}.git"
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception as e:
        print("Secrets:", e)
    if not token:
        token = getpass.getpass("GITHUB_TOKEN: ").strip()
    if not token:
        raise RuntimeError("Thiếu GITHUB_TOKEN")
    return f"https://{token}@github.com/{REPO_SLUG}.git"

if IN_COLAB:
    r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
    if r.returncode != 0 or not r.stdout.strip():
        raise RuntimeError("Cần GPU T4")
    print("GPU:", r.stdout.strip())
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    if not (COLAB_REPO_DIR / "SwiftEdit" / "infer.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", _colab_repo_url(), str(COLAB_REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(COLAB_REPO_DIR), "pull", "--ff-only"], check=False)
    PROJECT_ROOT = COLAB_REPO_DIR
    os.chdir(PROJECT_ROOT)
    os.environ.setdefault("HF_HOME", "/content/huggingface")
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent
print("PROJECT_ROOT", PROJECT_ROOT)

FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'

### ② Setup (pip + HF; **không** tải Qualcomm nếu SKIP)

In [ ]:
env = os.environ.copy()
env["COLAB_REPO_DIR"] = str(PROJECT_ROOT) if IN_COLAB else ""
if IN_COLAB and SKIP_QUALCOMM_IN_SETUP:
    env["SWIFTEDIT_SKIP_WEIGHTS_DOWNLOAD"] = "1"
setup_sh = PROJECT_ROOT / "scripts" / ("setup_colab.sh" if IN_COLAB else "setup_macos.sh")
subprocess.run(["bash", str(setup_sh)], check=True, env=env, cwd=PROJECT_ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "torchmetrics", "pyarrow"], check=True)

se = PROJECT_ROOT / "SwiftEdit"
# Dọn leftover cũ nếu còn (.part / .tar.gz)
removed = []
for p in list(se.glob("swiftedit_weights.tar.gz*")):
    print("rm leftover", p.name, p.stat().st_size // (1024**2), "MB")
    p.unlink(missing_ok=True)
    removed.append(p.name)
print("Removed archives:", removed or "(none)")
WP32 = se / "swiftedit_weights"
WP16 = se / "swiftedit_weights_fp16"

### ②b Dataset June17

In [ ]:
auto = PROJECT_ROOT / "data" / "PIE-Bench-auto200"
if not (auto / "annotation_images").is_dir():
    subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "freeze_piebench_auto200.py")], check=True, cwd=PROJECT_ROOT)
subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "build_june17_jobs.py")], check=True, cwd=PROJECT_ROOT)
print("jobs_june17:", (PROJECT_ROOT / "data" / "jobs_june17.json").is_file())

### ③ Prepare weights (Drive trước — tránh kéo 50GB)

Thứ tự: **Drive fp16/fp32 → symlink** → chỉ tải Qualcomm nếu configs cần fp32 và Drive trống → convert chỉ khi cho phép.

- Chỉ `*_weight` + Drive fp16 sẵn → **0 GB** Qualcomm.
- Cần fp32 mà phải tải: stream `curl|tar` (~10GB extract, **không** giữ .part/.tar).

In [ ]:
DRIVE_FP16 = Path(DRIVE_FP16)
DRIVE_FP32 = Path(DRIVE_FP32)
cmd = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "prepare_colab_weights.py"),
    "--configs", EVAL_CONFIGS,
    "--drive-fp16", str(DRIVE_FP16),
    "--drive-fp32", str(DRIVE_FP32),
    "--local-fp32", str(WP32),
    "--local-fp16", str(WP16),
]
if NO_CONVERT_ON_COLAB:
    cmd.append("--no-convert")
# Chỉ cho tải Qualcomm khi config cần fp32 (prepare tự quyết) — không khi đã có Drive
r = subprocess.run(cmd, cwd=PROJECT_ROOT)
if r.returncode != 0:
    raise RuntimeError(
        "prepare_colab_weights thất bại.\n"
        f"- Upload fp16 lên Drive: {DRIVE_FP16}\n"
        f"- Hoặc fp32: {DRIVE_FP32}\n"
        "- Hoặc set NO_CONVERT_ON_COLAB=False / SKIP_QUALCOMM_IN_SETUP=False"
    )
print("weights OK")
print("fp16:", (WP16 / "sbv2_0.5").is_dir(), "fp32:", (WP32 / "sbv2_0.5").is_dir())

### ④ Eval (chỉ configs đã chọn) → bundle.zip

In [ ]:
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
cmd = [
    sys.executable, "-u",
    str(PROJECT_ROOT / "scripts" / "run_precision_disk_vram_eval.py"),
    "--configs", EVAL_CONFIGS,
    "--n-images", str(N_IMAGES),
    "--edits-per-image", str(EDITS_PER_IMAGE),
    "--jobs-manifest", str(PROJECT_ROOT / "data" / "jobs_june17.json"),
    "--weights-fp32", str(WP32),
    "--weights-fp16", str(WP16),
]
if MAX_JOBS is not None:
    cmd += ["--max-jobs", str(MAX_JOBS)]
print(" ".join(cmd))
r = subprocess.run(cmd, cwd=PROJECT_ROOT, env=env)
print("exit", r.returncode)
if r.returncode != 0:
    raise RuntimeError(f"Eval exit {r.returncode}")
bundles = sorted((PROJECT_ROOT / "experimental_data").glob("precision_run_*/bundle.zip"))
print("Latest bundles:", bundles[-3:])
if IN_COLAB and bundles:
    from google.colab import files
    files.download(str(bundles[-1]))

### Sau khi đủ bundle (local)

```bash
# giải nén từng bundle.zip vào experimental_data/
python scripts/compare_precision_runs.py
# → experimental_data/PRECISION_FINAL_REPORT.md
```